# Relatório Unificado 3x3 - Top Ranks

Este notebook gera um único HTML chamado `3x3_visu_scores.html`, contendo os relatórios dos ranks 1, 2 e 3 em sequência. Também exporta tabelas `.tex` e matrizes `.pdf`/`.png` para a pasta `3x3_visu_scores/`.

Nesta versão, os títulos repetidos acima dos gráficos Plotly foram removidos: o título fica dentro do próprio gráfico, e não duplicado como cabeçalho HTML.

In [1]:

# ============================================================
# RELATÓRIO UNIFICADO 3x3 - TOP RANKS
# HTML + DOWNLOADS + EXPORTAÇÃO PARA LaTeX
#
# Entradas:
# - creditcard.csv
# - 3x3_scores.csv
#
# Saídas:
# - 3x3_visu_scores.html
# - 3x3_visu_scores/
#     ├── tabela_metricas_ranks.tex
#     ├── tabela_matrizes_confusao_ranks.tex
#     ├── comandos_latex_exemplo.tex
#     ├── matrizes .png/.pdf
#     └── gráficos Plotly .png/.pdf se o pacote kaleido estiver instalado
#
# Observação:
# - No HTML, os gráficos Plotly já têm botão nativo de download PNG na barra superior.
# - As matrizes de confusão também têm botão próprio "Baixar PNG".
# ============================================================

import io
import re
import html
import base64
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings("ignore")


# ============================================================
# CONFIGURAÇÕES
# ============================================================

ARQUIVO_DADOS = "creditcard.csv"
ARQUIVO_SCORES = "3x3_scores.csv"

PASTA_SAIDA = "."
ARQUIVO_HTML_SAIDA = "3x3_visu_scores.html"
PASTA_LATEX = "3x3_visu_scores"

RANKS = (1, 2, 3)

RAIO_ELIPSOIDE = 3.0

# Para o HTML não ficar absurdamente pesado.
# Fraudes são mantidas todas; não fraudes podem ser amostradas apenas nos gráficos interativos.
MAX_NAO_FRAUDE_INTERATIVO = 60000
RANDOM_STATE_AMOSTRA_HTML = 42

GMM_N_COMPONENTS = 2
GMM_COVARIANCE_TYPE = "full"
GMM_RANDOM_STATE = 42
GMM_N_INIT = 3
GMM_REG_COVAR = 1e-6

AZUL_NAO_FRAUDE = "#2563eb"
AZUL_NAO_FRAUDE_BORDA = "#1e3a8a"
AMARELO_FRAUDE = "#facc15"
VERMELHO_ERRO = "#dc2626"
VERMELHO_ESCURO = "#7f1d1d"
PRETO_BORDA = "#111827"


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def sanitizar_nome(nome):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(nome))


def formatar_count(valor):
    return f"{int(valor):,}".replace(",", ".")


def formatar_float_html(valor, casas=6):
    if pd.isna(valor):
        return "-"
    try:
        return f"{float(valor):.{casas}f}".replace(".", ",")
    except Exception:
        return str(valor)


def formatar_float_latex(valor, casas=6):
    if pd.isna(valor):
        return "-"
    try:
        return f"{float(valor):.{casas}f}"
    except Exception:
        return str(valor)


def fig_to_base64(fig, dpi=150):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("utf-8")


def salvar_figura_matplotlib(fig, caminho_sem_extensao):
    caminho_sem_extensao = Path(caminho_sem_extensao)
    fig.savefig(caminho_sem_extensao.with_suffix(".png"), dpi=300, bbox_inches="tight")
    fig.savefig(caminho_sem_extensao.with_suffix(".pdf"), bbox_inches="tight")


def exportar_plotly(fig, caminho_sem_extensao):
    """
    Exporta figuras Plotly em PNG/PDF quando o pacote kaleido está instalado.
    Se não estiver instalado, o HTML continua funcionando e o usuário ainda consegue
    baixar PNG pelo botão nativo do Plotly dentro do HTML.
    """
    caminho_sem_extensao = Path(caminho_sem_extensao)
    try:
        fig.write_image(str(caminho_sem_extensao.with_suffix(".png")), scale=2)
        fig.write_image(str(caminho_sem_extensao.with_suffix(".pdf")))
        return True
    except Exception as e:
        print(f"[Aviso] Não foi possível exportar {caminho_sem_extensao.name} como PNG/PDF via Plotly/kaleido: {e}")
        return False


def fig_plotly_to_html(fig, filename="grafico_3x3"):
    return pio.to_html(
        fig,
        full_html=False,
        include_plotlyjs=False,
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "responsive": True,
            "displaylogo": False,
            "toImageButtonOptions": {
                "format": "png",
                "filename": filename,
                "height": 900,
                "width": 1400,
                "scale": 2
            },
            "modeBarButtonsToRemove": ["lasso2d", "select2d"]
        }
    )


def fig_plotly_to_html_com_titulo(fig, titulo, filename="grafico_3x3"):
    """
    Coloca o título dentro do próprio gráfico Plotly e evita repetir
    o mesmo título como <h2> acima do gráfico no HTML.
    """
    fig.update_layout(
        title=dict(
            text=str(titulo),
            x=0.5,
            xanchor="center",
            font=dict(size=22)
        )
    )

    return fig_plotly_to_html(fig, filename=filename)


def gerar_elipsoide_3d_media_cov(media, cov, raio=3.0, n_u=72, n_v=36):
    autovalores, autovetores = np.linalg.eigh(cov)
    autovalores = np.maximum(autovalores, 1e-12)

    u = np.linspace(0, 2 * np.pi, n_u)
    v = np.linspace(0, np.pi, n_v)

    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))

    esfera = np.stack([x, y, z], axis=-1)
    transformacao = autovetores @ np.diag(np.sqrt(autovalores) * raio)
    elipsoide = esfera @ transformacao.T + media

    return elipsoide[:, :, 0], elipsoide[:, :, 1], elipsoide[:, :, 2]


def amostrar_nao_fraude_para_html(df_plot, target_name):
    if MAX_NAO_FRAUDE_INTERATIVO is None:
        return df_plot.copy()

    df_fraude = df_plot[df_plot[target_name] == 1].copy()
    df_nao_fraude = df_plot[df_plot[target_name] == 0].copy()

    if len(df_nao_fraude) > MAX_NAO_FRAUDE_INTERATIVO:
        df_nao_fraude = df_nao_fraude.sample(
            n=MAX_NAO_FRAUDE_INTERATIVO,
            random_state=RANDOM_STATE_AMOSTRA_HTML
        )

    return pd.concat([df_nao_fraude, df_fraude], axis=0).copy()


# ============================================================
# TABELAS HTML / LATEX
# ============================================================

def gerar_tabela_html(df, table_id, classe="data-table"):
    if df is None or df.empty:
        return "<p>Nenhum dado disponível.</p>"

    s = f'<table id="{html.escape(str(table_id))}" class="{classe}">\n<thead><tr>'

    for col in df.columns:
        s += f"<th>{html.escape(str(col))}</th>"

    s += "</tr></thead><tbody>\n"

    for _, row in df.iterrows():
        s += "<tr>"
        for valor in row:
            s += f"<td>{html.escape(str(valor))}</td>"
        s += "</tr>\n"

    s += "</tbody></table>"
    return s


def gerar_secao_tabela(titulo, tabela_html, table_id, nome_csv):
    return f"""
    <section class="plot-card">
        <div class="section-header">
            <h2>{html.escape(str(titulo))}</h2>
            <button class="download-btn" onclick="baixarTabelaCSV('{html.escape(str(table_id))}', '{html.escape(str(nome_csv))}')">
                Baixar CSV
            </button>
        </div>
        <div class="table-wrapper">
            {tabela_html}
        </div>
    </section>
    """


def preparar_df_latex(df):
    df_latex = df.copy()

    for col in df_latex.columns:
        if pd.api.types.is_float_dtype(df_latex[col]):
            df_latex[col] = df_latex[col].apply(lambda x: formatar_float_latex(x, 6))
        elif pd.api.types.is_integer_dtype(df_latex[col]):
            df_latex[col] = df_latex[col].apply(lambda x: int(x) if not pd.isna(x) else x)

    return df_latex


def salvar_tabela_latex(df, caminho, caption, label, longtable=False):
    caminho = Path(caminho)

    if df is None or df.empty:
        caminho.write_text("% Tabela vazia.\n", encoding="utf-8")
        return

    tex = preparar_df_latex(df).to_latex(
        index=False,
        escape=True,
        longtable=longtable,
        caption=caption,
        label=label
    )

    caminho.write_text(tex, encoding="utf-8")


def exportar_tabelas_latex(pasta_latex, tabela_metricas, tabela_matrizes):
    pasta_latex = Path(pasta_latex)
    pasta_latex.mkdir(parents=True, exist_ok=True)

    salvar_tabela_latex(
        tabela_metricas,
        pasta_latex / "tabela_metricas_ranks.tex",
        caption="Métricas dos melhores rankings do experimento 3x3.",
        label="tab:metricas-ranks-3x3",
        longtable=False
    )

    salvar_tabela_latex(
        tabela_matrizes,
        pasta_latex / "tabela_matrizes_confusao_ranks.tex",
        caption="Matrizes de confusão dos melhores rankings do experimento 3x3.",
        label="tab:matrizes-confusao-ranks-3x3",
        longtable=True
    )

    comandos_latex = r"""
% ============================================================
% COMANDOS LaTeX DE EXEMPLO
%
% Pacotes recomendados:
% \usepackage{graphicx}
% \usepackage{float}
% \usepackage{booktabs}
% \usepackage{longtable}
% \usepackage{pdflscape}
% ============================================================

\begin{table}[H]
\centering
\caption{Métricas dos melhores rankings do experimento 3x3.}
\label{tab:metricas-ranks-3x3-main}
\input{3x3_visu_scores/tabela_metricas_ranks.tex}
\end{table}

\begin{landscape}
\small
\input{3x3_visu_scores/tabela_matrizes_confusao_ranks.tex}
\end{landscape}

% Exemplo de figura:
% Troque FEATURE1, FEATURE2 e FEATURE3 pelo nome gerado no arquivo.
\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{3x3_visu_scores/rank_1_FEATURE1__FEATURE2__FEATURE3_matriz_melhor_corte.pdf}
\caption{Matriz de confusão do melhor ponto de corte do Rank 1.}
\label{fig:rank1-3x3-matriz}
\end{figure}
"""

    (pasta_latex / "comandos_latex_exemplo.tex").write_text(comandos_latex, encoding="utf-8")


# ============================================================
# MATRIZ DE CONFUSÃO
# ============================================================

def gerar_matriz_confusao(y_real, probabilidades, threshold):
    y_pred = (probabilidades >= threshold).astype(int)
    return confusion_matrix(y_real, y_pred, labels=[0, 1])


def preparar_valores_matriz(cm):
    tn, fp, fn, tp = cm.ravel()

    total_fraudes = fn + tp
    total_nao_fraudes = tn + fp

    fn_pct = fn / total_fraudes * 100 if total_fraudes != 0 else 0
    tp_pct = tp / total_fraudes * 100 if total_fraudes != 0 else 0
    tn_pct = tn / total_nao_fraudes * 100 if total_nao_fraudes != 0 else 0
    fp_pct = fp / total_nao_fraudes * 100 if total_nao_fraudes != 0 else 0

    return {
        "fn": {"pct": fn_pct, "count": int(fn), "qualidade": 100 - fn_pct},
        "tp": {"pct": tp_pct, "count": int(tp), "qualidade": tp_pct},
        "tn": {"pct": tn_pct, "count": int(tn), "qualidade": tn_pct},
        "fp": {"pct": fp_pct, "count": int(fp), "qualidade": 100 - fp_pct},
        "raw": {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}
    }


def preparar_valores_matriz_ideal(y_real):
    y_real_array = np.asarray(y_real).astype(int)

    total_fraudes = int(np.sum(y_real_array == 1))
    total_nao_fraudes = int(np.sum(y_real_array == 0))

    return {
        "fn": {"pct": 0.0, "count": 0, "qualidade": 100.0},
        "tp": {"pct": 100.0, "count": total_fraudes, "qualidade": 100.0},
        "tn": {"pct": 100.0, "count": total_nao_fraudes, "qualidade": 100.0},
        "fp": {"pct": 0.0, "count": 0, "qualidade": 100.0},
        "raw": {"TN": total_nao_fraudes, "FP": 0, "FN": 0, "TP": total_fraudes}
    }


def cor_por_qualidade(q):
    if q >= 95:
        return "cell q95"
    elif q >= 85:
        return "cell q85"
    elif q >= 70:
        return "cell q70"
    elif q >= 50:
        return "cell q50"
    elif q >= 30:
        return "cell q30"
    return "cell q10"


def gerar_html_matriz(titulo, valores, matriz_ideal=False, imagem_base64=None, nome_imagem=None):
    if matriz_ideal:
        desc_fn = "Erro ideal: nenhuma fraude perdida"
        desc_tp = "Acerto ideal: fraudes detectadas"
        desc_tn = "Acerto ideal: não fraudes corretas"
        desc_fp = "Erro ideal: nenhum falso alerta"
    else:
        desc_fn = "Erro: fraude perdida"
        desc_tp = "Acerto: fraude detectada"
        desc_tn = "Acerto: não fraude"
        desc_fp = "Erro: falso alerta"

    botao = ""

    if imagem_base64 is not None and nome_imagem is not None:
        botao = f"""
        <div class="section-actions-only">
            <a class="download-btn link-btn" href="data:image/png;base64,{imagem_base64}" download="{html.escape(str(nome_imagem))}">
                Baixar PNG
            </a>
        </div>
        """

    return f"""
    <section class="matrix-card">
        <h2>{html.escape(str(titulo))}</h2>
        {botao}
        <div class="matrix-area">
            <div class="matrix-wrapper">
                <div class="corner"></div>
                <div class="x-label">Pred Não Fraude</div>
                <div class="x-label">Pred Fraude</div>

                <div class="y-label">Real Fraude</div>

                <div class="{cor_por_qualidade(valores['fn']['qualidade'])}">
                    <div class="pct">{valores['fn']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['fn']['count'])})</div>
                    <div class="cell-desc">{desc_fn}</div>
                </div>

                <div class="{cor_por_qualidade(valores['tp']['qualidade'])}">
                    <div class="pct">{valores['tp']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['tp']['count'])})</div>
                    <div class="cell-desc">{desc_tp}</div>
                </div>

                <div class="y-label">Real Não Fraude</div>

                <div class="{cor_por_qualidade(valores['tn']['qualidade'])}">
                    <div class="pct">{valores['tn']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['tn']['count'])})</div>
                    <div class="cell-desc">{desc_tn}</div>
                </div>

                <div class="{cor_por_qualidade(valores['fp']['qualidade'])}">
                    <div class="pct">{valores['fp']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['fp']['count'])})</div>
                    <div class="cell-desc">{desc_fp}</div>
                </div>
            </div>

            <div class="legend">
                <div class="legend-title">Qualidade</div>
                <div class="colorbar"></div>
                <div class="legend-label-top">Melhor</div>
                <div class="legend-label-bottom">Pior</div>
            </div>
        </div>
    </section>
    """


def gerar_fig_matriz_confusao(titulo, valores):
    matriz_pct = np.array([
        [valores["fn"]["pct"], valores["tp"]["pct"]],
        [valores["tn"]["pct"], valores["fp"]["pct"]]
    ])

    matriz_count = np.array([
        [valores["fn"]["count"], valores["tp"]["count"]],
        [valores["tn"]["count"], valores["fp"]["count"]]
    ])

    fig, ax = plt.subplots(figsize=(8.5, 6.2))

    im = ax.imshow(matriz_pct, vmin=0, vmax=100, cmap="Blues")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Percentual por classe real (%)", fontweight="bold")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred Não Fraude", "Pred Fraude"], fontweight="bold")
    ax.set_yticklabels(["Real Fraude", "Real Não Fraude"], fontweight="bold")
    ax.set_title(titulo, fontsize=14, fontweight="bold", pad=14)

    textos = [
        ["Fraude perdida", "Fraude detectada"],
        ["Não fraude correta", "Falso alerta"]
    ]

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                f"{matriz_pct[i, j]:.2f}%\n({formatar_count(matriz_count[i, j])})\n{textos[i][j]}",
                ha="center",
                va="center",
                color="black",
                fontweight="bold",
                fontsize=10
            )

    plt.tight_layout()
    return fig


# ============================================================
# GRÁFICOS PLOTLY
# ============================================================

def gerar_grafico_3d_classe_real(df_plot, features, target_name):
    f1, f2, f3 = features

    df_html = amostrar_nao_fraude_para_html(df_plot, target_name)

    dados_nao_fraude = df_html.loc[df_html[target_name] == 0, features].dropna()
    dados_fraude = df_html.loc[df_html[target_name] == 1, features].dropna()

    total_nao_fraude = int((df_plot[target_name] == 0).sum())
    total_fraude = int((df_plot[target_name] == 1).sum())

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=dados_nao_fraude[f1],
            y=dados_nao_fraude[f2],
            z=dados_nao_fraude[f3],
            mode="markers",
            name=f"Não Fraude exibida ({formatar_count(len(dados_nao_fraude))} de {formatar_count(total_nao_fraude)})",
            marker=dict(size=2.7, color=AZUL_NAO_FRAUDE, opacity=0.14),
            hovertemplate=(
                f"{f1}: %{{x:.4f}}<br>"
                f"{f2}: %{{y:.4f}}<br>"
                f"{f3}: %{{z:.4f}}<br>"
                "Classe: Não Fraude"
                "<extra></extra>"
            )
        )
    )

    fig.add_trace(
        go.Scatter3d(
            x=dados_fraude[f1],
            y=dados_fraude[f2],
            z=dados_fraude[f3],
            mode="markers",
            name=f"Fraude ({formatar_count(total_fraude)})",
            marker=dict(
                size=7.5,
                color=AMARELO_FRAUDE,
                opacity=0.98,
                line=dict(color=PRETO_BORDA, width=1.2)
            ),
            hovertemplate=(
                f"{f1}: %{{x:.4f}}<br>"
                f"{f2}: %{{y:.4f}}<br>"
                f"{f3}: %{{z:.4f}}<br>"
                "Classe: Fraude"
                "<extra></extra>"
            )
        )
    )

    fig.update_layout(
        height=760,
        margin=dict(l=0, r=0, t=35, b=0),
        legend=dict(
            title="Classe Real",
            x=0.58,
            y=0.96,
            bgcolor="rgba(255,255,255,0.92)",
            bordercolor="#d1d5db",
            borderwidth=1,
            font=dict(size=13),
            itemsizing="constant"
        ),
        scene=dict(
            xaxis_title=f1,
            yaxis_title=f2,
            zaxis_title=f3,
            xaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
            yaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
            zaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
            camera=dict(eye=dict(x=1.65, y=1.45, z=0.95))
        )
    )

    return fig


def gerar_boxplots_3_features(df_plot, features, target_name):
    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=[
            f"Boxplot - {features[0]}",
            f"Boxplot - {features[1]}",
            f"Boxplot - {features[2]}"
        ],
        horizontal_spacing=0.08
    )

    for idx, feature in enumerate(features, start=1):
        dados_nao_fraude = df_plot.loc[df_plot[target_name] == 0, feature].dropna()
        dados_fraude = df_plot.loc[df_plot[target_name] == 1, feature].dropna()

        fig.add_trace(
            go.Box(
                y=dados_nao_fraude,
                name="Não Fraude",
                marker_color=AZUL_NAO_FRAUDE,
                boxmean=True,
                opacity=0.75,
                showlegend=True if idx == 1 else False
            ),
            row=1,
            col=idx
        )

        fig.add_trace(
            go.Box(
                y=dados_fraude,
                name="Fraude",
                marker_color=AMARELO_FRAUDE,
                boxmean=True,
                opacity=0.90,
                showlegend=True if idx == 1 else False
            ),
            row=1,
            col=idx
        )

        fig.update_yaxes(
            title_text=feature,
            row=1,
            col=idx,
            showgrid=True,
            gridcolor="rgba(148,163,184,0.28)",
            zeroline=False
        )

    fig.update_layout(
        height=560,
        margin=dict(l=50, r=40, t=95, b=60),
        legend=dict(
            title="Classe Real",
            orientation="h",
            x=0.5,
            y=1.14,
            xanchor="center",
            yanchor="bottom",
            bgcolor="rgba(255,255,255,0.92)",
            bordercolor="#d1d5db",
            borderwidth=1
        ),
        boxmode="group",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    return fig


def gerar_matriz_spearman_3d(df_plot, features, target_name):
    dados_corr = df_plot[features + [target_name]].copy()
    dados_corr = dados_corr.rename(columns={target_name: "Fraude"})

    corr = dados_corr.corr(method="spearman")
    labels = corr.columns.tolist()

    fig = go.Figure(
        data=go.Heatmap(
            z=corr.values,
            x=labels,
            y=labels,
            zmin=-1,
            zmax=1,
            colorscale="RdBu",
            reversescale=True,
            colorbar=dict(title=dict(text="Spearman", side="top")),
            text=np.round(corr.values, 3),
            texttemplate="%{text}",
            hovertemplate=(
                "Linha: %{y}<br>"
                "Coluna: %{x}<br>"
                "Spearman: %{z:.6f}"
                "<extra></extra>"
            )
        )
    )

    fig.update_layout(
        height=620,
        margin=dict(l=80, r=40, t=40, b=80),
        xaxis=dict(tickangle=-35),
        yaxis=dict(autorange="reversed")
    )

    return fig


def gerar_grafico_3d_elipsoide(df_plot, features, target_name, scaler, gmm, cluster_nao_fraude, threshold, raio_elipsoide=3.0):
    f1, f2, f3 = features

    df_html = amostrar_nao_fraude_para_html(df_plot, target_name)

    y_real_array = df_html[target_name].astype(int).to_numpy()
    resp = df_html["Responsabilidade_GMM_Fraude"].to_numpy()

    mask_nao_fraude = y_real_array == 0
    mask_fraude = y_real_array == 1

    mask_dentro = resp < threshold
    mask_fora = resp >= threshold

    media_scaled = gmm.means_[cluster_nao_fraude]
    cov_scaled = gmm.covariances_[cluster_nao_fraude]

    scale = scaler.scale_
    mean_scaler = scaler.mean_

    media_original = media_scaled * scale + mean_scaler
    matriz_scale = np.diag(scale)
    cov_original = matriz_scale @ cov_scaled @ matriz_scale

    ell_x, ell_y, ell_z = gerar_elipsoide_3d_media_cov(
        media=media_original,
        cov=cov_original,
        raio=raio_elipsoide,
        n_u=72,
        n_v=36
    )

    fig = go.Figure()

    fig.add_trace(
        go.Surface(
            x=ell_x,
            y=ell_y,
            z=ell_z,
            opacity=0.22,
            colorscale=[[0.0, AZUL_NAO_FRAUDE], [1.0, AZUL_NAO_FRAUDE]],
            showscale=False,
            name="Elipsoide do componente não fraude"
        )
    )

    for k in np.linspace(0, ell_x.shape[1] - 1, 7).astype(int):
        fig.add_trace(
            go.Scatter3d(
                x=ell_x[:, k],
                y=ell_y[:, k],
                z=ell_z[:, k],
                mode="lines",
                line=dict(color=AZUL_NAO_FRAUDE_BORDA, width=4),
                name="Contorno da elipsoide" if k == 0 else None,
                showlegend=True if k == 0 else False,
                hoverinfo="skip"
            )
        )

    grupos = [
        (mask_nao_fraude & mask_dentro, "Não fraude dentro", AZUL_NAO_FRAUDE, 2.6, 0.09, None),
        (mask_fraude & mask_dentro, "Fraude dentro", AMARELO_FRAUDE, 6.3, 0.62, PRETO_BORDA),
        (mask_nao_fraude & mask_fora, "Não fraude fora", VERMELHO_ERRO, 5.4, 0.84, VERMELHO_ESCURO),
        (mask_fraude & mask_fora, "Fraude fora", AMARELO_FRAUDE, 8.8, 0.98, PRETO_BORDA),
    ]

    for mask, nome, cor, tamanho, opacidade, borda in grupos:
        marker = dict(size=tamanho, color=cor, opacity=opacidade)
        if borda is not None:
            marker["line"] = dict(color=borda, width=1.0)

        fig.add_trace(
            go.Scatter3d(
                x=df_html.loc[mask, f1],
                y=df_html.loc[mask, f2],
                z=df_html.loc[mask, f3],
                mode="markers",
                name=f"{nome} ({formatar_count(mask.sum())})",
                marker=marker
            )
        )

    fig.update_layout(
        height=840,
        margin=dict(l=0, r=30, t=40, b=0),
        legend=dict(
            title="Legenda",
            x=0.52,
            y=0.97,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.92)",
            bordercolor="#d1d5db",
            borderwidth=1,
            font=dict(size=13),
            itemsizing="constant"
        ),
        scene=dict(
            xaxis_title=f1,
            yaxis_title=f2,
            zaxis_title=f3,
            xaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
            yaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
            zaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
            camera=dict(eye=dict(x=1.65, y=1.45, z=0.95))
        )
    )

    return fig


def gerar_responsabilidades_3d_projecoes(df_plot, features, target_name, scaler, gmm, cluster_fraude, threshold):
    f1, f2, f3 = features

    medianas = {
        f1: df_plot[f1].median(),
        f2: df_plot[f2].median(),
        f3: df_plot[f3].median()
    }

    pares = [
        (f1, f2, f3),
        (f1, f3, f2),
        (f2, f3, f1)
    ]

    df_html = amostrar_nao_fraude_para_html(df_plot, target_name)

    y_real_array = df_html[target_name].astype(int).to_numpy()
    mask_nao_fraude = y_real_array == 0
    mask_fraude = y_real_array == 1

    colorscale_resp_2d = [
        [0.00, "#3358c9"],
        [0.12, "#5f7fe0"],
        [0.25, "#b9c8f2"],
        [0.40, "#e4e9f8"],
        [0.50, "#f7f7f7"],
        [0.62, "#f6dfd9"],
        [0.75, "#efc1ba"],
        [0.88, "#e7a9ad"],
        [1.00, "#d88997"]
    ]

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=[
            f"{f1} x {f2}",
            f"{f1} x {f3}",
            f"{f2} x {f3}",
        ],
        horizontal_spacing=0.07
    )

    for idx, (eixo_x, eixo_y, fixo) in enumerate(pares, start=1):
        x_min = df_plot[eixo_x].min()
        x_max = df_plot[eixo_x].max()
        y_min = df_plot[eixo_y].min()
        y_max = df_plot[eixo_y].max()

        margem_x = 0.05 * (x_max - x_min)
        margem_y = 0.05 * (y_max - y_min)

        x_grid = np.linspace(x_min - margem_x, x_max + margem_x, 220)
        y_grid = np.linspace(y_min - margem_y, y_max + margem_y, 220)

        xx, yy = np.meshgrid(x_grid, y_grid)

        grid_df = pd.DataFrame(
            {
                f1: medianas[f1],
                f2: medianas[f2],
                f3: medianas[f3]
            },
            index=np.arange(xx.size)
        )

        grid_df[eixo_x] = xx.ravel()
        grid_df[eixo_y] = yy.ravel()
        grid_df[fixo] = medianas[fixo]
        grid_df = grid_df[features]

        grid_scaled = scaler.transform(grid_df)
        grid_prob = gmm.predict_proba(grid_scaled)[:, cluster_fraude]
        zz = grid_prob.reshape(xx.shape)

        fig.add_trace(
            go.Contour(
                x=x_grid,
                y=y_grid,
                z=zz,
                colorscale=colorscale_resp_2d,
                zmin=0,
                zmax=1,
                contours=dict(
                    start=0,
                    end=1,
                    size=0.05,
                    coloring="heatmap",
                    showlabels=False
                ),
                opacity=0.62,
                showscale=True if idx == 3 else False,
                colorbar=dict(
                    title=dict(
                        text="Responsabilidade<br>GMM para fraude",
                        side="right",
                        font=dict(size=13)
                    ),
                    tickvals=[0, 0.25, 0.5, 0.75, 1],
                    ticktext=["0.00<br>Baixa", "0.25", "0.50", "0.75", "1.00<br>Alta"],
                    tickfont=dict(size=11),
                    len=0.74,
                    thickness=22,
                    x=1.03,
                    y=0.48,
                    yanchor="middle"
                ),
                showlegend=False
            ),
            row=1,
            col=idx
        )

        fig.add_trace(
            go.Contour(
                x=x_grid,
                y=y_grid,
                z=zz,
                contours=dict(
                    start=0.25,
                    end=0.75,
                    size=0.25,
                    coloring="none",
                    showlabels=False
                ),
                line=dict(color="#6b7280", width=1.1, dash="dot"),
                showscale=False,
                hoverinfo="skip",
                name="Curvas auxiliares: 0.25, 0.50 e 0.75",
                showlegend=True if idx == 1 else False
            ),
            row=1,
            col=idx
        )

        fig.add_trace(
            go.Contour(
                x=x_grid,
                y=y_grid,
                z=zz,
                contours=dict(
                    start=threshold,
                    end=threshold,
                    size=1,
                    coloring="none",
                    showlabels=False
                ),
                line=dict(color="#020617", width=3.2, dash="dash"),
                showscale=False,
                hoverinfo="skip",
                name="Curva de nível da GMM no ponto de corte",
                showlegend=True if idx == 1 else False
            ),
            row=1,
            col=idx
        )

        fig.add_trace(
            go.Scattergl(
                x=df_html.loc[mask_nao_fraude, eixo_x],
                y=df_html.loc[mask_nao_fraude, eixo_y],
                mode="markers",
                marker=dict(size=3.2, color=AZUL_NAO_FRAUDE, opacity=0.12),
                name="Não Fraude pontos",
                showlegend=False
            ),
            row=1,
            col=idx
        )

        fig.add_trace(
            go.Scattergl(
                x=df_html.loc[mask_fraude, eixo_x],
                y=df_html.loc[mask_fraude, eixo_y],
                mode="markers",
                marker=dict(
                    size=7.5,
                    color=AMARELO_FRAUDE,
                    opacity=0.98,
                    line=dict(color=PRETO_BORDA, width=1.1)
                ),
                name=f"Fraude real ({formatar_count(mask_fraude.sum())})",
                showlegend=True if idx == 1 else False
            ),
            row=1,
            col=idx
        )

        if idx == 1:
            fig.add_trace(
                go.Scattergl(
                    x=[None],
                    y=[None],
                    mode="markers",
                    name=f"Não Fraude exibida ({formatar_count(mask_nao_fraude.sum())})",
                    marker=dict(
                        size=9,
                        color=AZUL_NAO_FRAUDE,
                        opacity=1.0,
                        line=dict(color=AZUL_NAO_FRAUDE_BORDA, width=1.2)
                    ),
                    hoverinfo="skip",
                    showlegend=True
                ),
                row=1,
                col=idx
            )

        fig.update_xaxes(
            title_text=eixo_x,
            row=1,
            col=idx,
            showgrid=True,
            gridcolor="rgba(148,163,184,0.28)",
            zeroline=False
        )

        fig.update_yaxes(
            title_text=eixo_y,
            row=1,
            col=idx,
            showgrid=True,
            gridcolor="rgba(148,163,184,0.28)",
            zeroline=False
        )

    fig.update_layout(
        height=760,
        margin=dict(l=40, r=125, t=135, b=60),
        legend=dict(
            title="Legenda",
            orientation="h",
            x=0.5,
            y=1.10,
            xanchor="center",
            yanchor="bottom",
            bgcolor="rgba(255,255,255,0.92)",
            bordercolor="#d1d5db",
            borderwidth=1,
            font=dict(size=13),
            itemsizing="constant"
        ),
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    return fig


# ============================================================
# PROCESSAR RANK
# ============================================================

def processar_rank_3x3(rank, df, scores_3x3, target_name, pasta_latex, raio_elipsoide=RAIO_ELIPSOIDE):
    colunas_necessarias = [
        "Feature_1",
        "Feature_2",
        "Feature_3",
        "Posicao_Rank",
        "AUC_PR",
        "MCC",
        "Score_Final",
        "Melhor_Ponto_Corte"
    ]

    for coluna in colunas_necessarias:
        if coluna not in scores_3x3.columns:
            raise ValueError(f"O arquivo de scores precisa ter a coluna '{coluna}'.")

    if "Ponto_Corte_Medio" not in scores_3x3.columns:
        scores_3x3["Ponto_Corte_Medio"] = 0.5

    if rank not in scores_3x3["Posicao_Rank"].values:
        raise ValueError(
            f"Rank {rank} não encontrado. "
            f"Ranks disponíveis: 1 até {scores_3x3['Posicao_Rank'].max()}."
        )

    linha_rank = scores_3x3.loc[
        scores_3x3["Posicao_Rank"] == rank
    ].iloc[0]

    feature_1 = linha_rank["Feature_1"]
    feature_2 = linha_rank["Feature_2"]
    feature_3 = linha_rank["Feature_3"]

    features = [feature_1, feature_2, feature_3]

    melhor_ponto_corte = float(linha_rank["Melhor_Ponto_Corte"])
    ponto_corte_medio = float(linha_rank["Ponto_Corte_Medio"])

    auc_pr = float(linha_rank["AUC_PR"])
    mcc = float(linha_rank["MCC"])
    score_final = float(linha_rank["Score_Final"])

    if "Log_Loss_Norm" in linha_rank.index:
        log_loss_norm = float(linha_rank["Log_Loss_Norm"])
    elif "Log_Loss" in linha_rank.index:
        log_loss_norm = 1 / (1 + float(linha_rank["Log_Loss"]))
    else:
        log_loss_norm = np.nan

    if "Diferenca_Neg_Log_Veross" in linha_rank.index:
        diferenca_neg_log_veross = float(linha_rank["Diferenca_Neg_Log_Veross"])
    elif (
        "Neg_Log_Veross_Com_Rotulo" in linha_rank.index
        and "Neg_Log_Veross_GMM" in linha_rank.index
    ):
        diferenca_neg_log_veross = (
            float(linha_rank["Neg_Log_Veross_Com_Rotulo"])
            - float(linha_rank["Neg_Log_Veross_GMM"])
        )
    else:
        diferenca_neg_log_veross = np.nan

    temp = df[features + [target_name]].dropna().copy()

    X = temp[features]
    y_real = temp[target_name].astype(int)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    gmm = GaussianMixture(
        n_components=GMM_N_COMPONENTS,
        covariance_type=GMM_COVARIANCE_TYPE,
        random_state=GMM_RANDOM_STATE,
        n_init=GMM_N_INIT,
        reg_covar=GMM_REG_COVAR
    )

    gmm.fit(X_scaled)

    clusters = gmm.predict(X_scaled)
    ct = pd.crosstab(clusters, y_real)

    if 1 not in ct.columns:
        raise ValueError("A classe fraude, valor 1, não foi encontrada no target.")

    cluster_fraude = ct[1].idxmax()
    cluster_nao_fraude = 1 - cluster_fraude

    prob_fraude = gmm.predict_proba(X_scaled)[:, cluster_fraude]
    prob_fraude = np.clip(prob_fraude, 1e-15, 1 - 1e-15)

    temp["Responsabilidade_GMM_Fraude"] = prob_fraude

    cm_melhor = gerar_matriz_confusao(
        y_real=y_real,
        probabilidades=prob_fraude,
        threshold=melhor_ponto_corte
    )

    cm_medio = gerar_matriz_confusao(
        y_real=y_real,
        probabilidades=prob_fraude,
        threshold=ponto_corte_medio
    )

    valores_melhor = preparar_valores_matriz(cm_melhor)
    valores_medio = preparar_valores_matriz(cm_medio)
    valores_ideal = preparar_valores_matriz_ideal(y_real)

    prefixo = (
        f"rank_{rank}_"
        f"{sanitizar_nome(feature_1)}__"
        f"{sanitizar_nome(feature_2)}__"
        f"{sanitizar_nome(feature_3)}"
    )

    combinacao_nome = f"{feature_1} + {feature_2} + {feature_3}"

    # ========================================================
    # FIGURAS PLOTLY
    # ========================================================

    fig_classe_real = gerar_grafico_3d_classe_real(temp, features, target_name)
    fig_boxplots = gerar_boxplots_3_features(temp, features, target_name)
    fig_spearman = gerar_matriz_spearman_3d(temp, features, target_name)

    fig_elipsoide_melhor = gerar_grafico_3d_elipsoide(
        df_plot=temp,
        features=features,
        target_name=target_name,
        scaler=scaler,
        gmm=gmm,
        cluster_nao_fraude=cluster_nao_fraude,
        threshold=melhor_ponto_corte,
        raio_elipsoide=raio_elipsoide
    )

    fig_elipsoide_medio = gerar_grafico_3d_elipsoide(
        df_plot=temp,
        features=features,
        target_name=target_name,
        scaler=scaler,
        gmm=gmm,
        cluster_nao_fraude=cluster_nao_fraude,
        threshold=ponto_corte_medio,
        raio_elipsoide=raio_elipsoide
    )

    fig_projecoes_melhor = gerar_responsabilidades_3d_projecoes(
        df_plot=temp,
        features=features,
        target_name=target_name,
        scaler=scaler,
        gmm=gmm,
        cluster_fraude=cluster_fraude,
        threshold=melhor_ponto_corte
    )

    fig_projecoes_medio = gerar_responsabilidades_3d_projecoes(
        df_plot=temp,
        features=features,
        target_name=target_name,
        scaler=scaler,
        gmm=gmm,
        cluster_fraude=cluster_fraude,
        threshold=ponto_corte_medio
    )

    figs_plotly = {
        "dispersao_3d": fig_classe_real,
        "boxplots": fig_boxplots,
        "spearman": fig_spearman,
        "elipsoide_melhor_corte": fig_elipsoide_melhor,
        "elipsoide_corte_medio": fig_elipsoide_medio,
        "responsabilidades_melhor_corte": fig_projecoes_melhor,
        "responsabilidades_corte_medio": fig_projecoes_medio,
    }

    for nome, fig in figs_plotly.items():
        exportar_plotly(
            fig,
            Path(pasta_latex) / f"{prefixo}_{nome}"
        )

    # ========================================================
    # MATRIZES ESTÁTICAS
    # ========================================================

    figs_matriz = {
        "matriz_melhor_corte": gerar_fig_matriz_confusao(
            f"Rank {rank} - {combinacao_nome} - Melhor Ponto de Corte",
            valores_melhor
        ),
        "matriz_corte_medio": gerar_fig_matriz_confusao(
            f"Rank {rank} - {combinacao_nome} - Ponto de Corte 0.5",
            valores_medio
        ),
        "matriz_ideal": gerar_fig_matriz_confusao(
            f"Rank {rank} - {combinacao_nome} - Matriz Ideal",
            valores_ideal
        )
    }

    imagens_matriz_base64 = {}

    for nome, fig in figs_matriz.items():
        nome_arquivo = f"{prefixo}_{nome}"
        salvar_figura_matplotlib(
            fig,
            Path(pasta_latex) / nome_arquivo
        )
        imagens_matriz_base64[nome] = fig_to_base64(fig)
        plt.close(fig)

    # ========================================================
    # TABELAS DO RANK
    # ========================================================

    tabela_metricas_rank = pd.DataFrame([
        {
            "Rank": rank,
            "Feature_1": feature_1,
            "Feature_2": feature_2,
            "Feature_3": feature_3,
            "Melhor_Ponto_Corte": melhor_ponto_corte,
            "Ponto_Corte_Medio": ponto_corte_medio,
            "AUC_PR": auc_pr,
            "MCC": mcc,
            "Log_Loss_Norm": log_loss_norm,
            "Score_Final": score_final,
            "Diferenca_Neg_Log_Veross": diferenca_neg_log_veross,
            "GMM_N_Components": GMM_N_COMPONENTS,
            "GMM_Covariance_Type": GMM_COVARIANCE_TYPE,
            "GMM_Random_State": GMM_RANDOM_STATE,
            "GMM_N_Init": GMM_N_INIT,
            "GMM_Reg_Covar": GMM_REG_COVAR,
            "Cluster_Fraude": int(cluster_fraude),
            "Cluster_Nao_Fraude": int(cluster_nao_fraude),
            "Raio_Elipsoide": raio_elipsoide
        }
    ])

    tabela_matrizes_rank = pd.DataFrame([
        {
            "Rank": rank,
            "Feature_1": feature_1,
            "Feature_2": feature_2,
            "Feature_3": feature_3,
            "Cenario": "Melhor ponto de corte",
            "Threshold": melhor_ponto_corte,
            **valores_melhor["raw"],
            "FN_Pct_Real_Fraude": valores_melhor["fn"]["pct"],
            "TP_Pct_Real_Fraude": valores_melhor["tp"]["pct"],
            "TN_Pct_Real_Nao_Fraude": valores_melhor["tn"]["pct"],
            "FP_Pct_Real_Nao_Fraude": valores_melhor["fp"]["pct"]
        },
        {
            "Rank": rank,
            "Feature_1": feature_1,
            "Feature_2": feature_2,
            "Feature_3": feature_3,
            "Cenario": "Ponto de corte médio",
            "Threshold": ponto_corte_medio,
            **valores_medio["raw"],
            "FN_Pct_Real_Fraude": valores_medio["fn"]["pct"],
            "TP_Pct_Real_Fraude": valores_medio["tp"]["pct"],
            "TN_Pct_Real_Nao_Fraude": valores_medio["tn"]["pct"],
            "FP_Pct_Real_Nao_Fraude": valores_medio["fp"]["pct"]
        },
        {
            "Rank": rank,
            "Feature_1": feature_1,
            "Feature_2": feature_2,
            "Feature_3": feature_3,
            "Cenario": "Ideal",
            "Threshold": np.nan,
            **valores_ideal["raw"],
            "FN_Pct_Real_Fraude": valores_ideal["fn"]["pct"],
            "TP_Pct_Real_Fraude": valores_ideal["tp"]["pct"],
            "TN_Pct_Real_Nao_Fraude": valores_ideal["tn"]["pct"],
            "FP_Pct_Real_Nao_Fraude": valores_ideal["fp"]["pct"]
        }
    ])

    tabela_metricas_html = tabela_metricas_rank.copy()

    for col in tabela_metricas_html.columns:
        if pd.api.types.is_float_dtype(tabela_metricas_html[col]):
            tabela_metricas_html[col] = tabela_metricas_html[col].apply(lambda x: formatar_float_html(x, 6))

    html_metricas_rank = gerar_secao_tabela(
        titulo=f"Métricas do Rank {rank}",
        tabela_html=gerar_tabela_html(tabela_metricas_html, table_id=f"tabela_metricas_rank_{rank}"),
        table_id=f"tabela_metricas_rank_{rank}",
        nome_csv=f"{prefixo}_metricas.csv"
    )

    html_melhor = gerar_html_matriz(
        titulo=f"Matriz de Confusão (%) - {combinacao_nome} - Melhor Ponto de Corte",
        valores=valores_melhor,
        imagem_base64=imagens_matriz_base64["matriz_melhor_corte"],
        nome_imagem=f"{prefixo}_matriz_melhor_corte.png"
    )

    html_medio = gerar_html_matriz(
        titulo=f"Matriz de Confusão (%) - {combinacao_nome} - Ponto de Corte 0.5",
        valores=valores_medio,
        imagem_base64=imagens_matriz_base64["matriz_corte_medio"],
        nome_imagem=f"{prefixo}_matriz_corte_medio.png"
    )

    html_ideal = gerar_html_matriz(
        titulo="Matriz de Confusão Ideal (%)",
        valores=valores_ideal,
        matriz_ideal=True,
        imagem_base64=imagens_matriz_base64["matriz_ideal"],
        nome_imagem=f"{prefixo}_matriz_ideal.png"
    )

    html_rank = f"""
    <section class="rank-section" id="rank-{rank}">
        <h1>Rank {rank} - Features {html.escape(str(feature_1))} + {html.escape(str(feature_2))} + {html.escape(str(feature_3))}</h1>

        <div class="info-box">
            <div class="info-grid">
                <div class="info-item"><div class="info-label">Rank</div><div class="info-value">{rank}</div></div>
                <div class="info-item"><div class="info-label">Feature 1</div><div class="info-value">{html.escape(str(feature_1))}</div></div>
                <div class="info-item"><div class="info-label">Feature 2</div><div class="info-value">{html.escape(str(feature_2))}</div></div>
                <div class="info-item"><div class="info-label">Feature 3</div><div class="info-value">{html.escape(str(feature_3))}</div></div>
                <div class="info-item"><div class="info-label">Melhor Ponto de Corte</div><div class="info-value">{melhor_ponto_corte:.6f}</div></div>
                <div class="info-item"><div class="info-label">Ponto de Corte Médio</div><div class="info-value">{ponto_corte_medio:.6f}</div></div>
                <div class="info-item"><div class="info-label">AUC-PR</div><div class="info-value">{auc_pr:.6f}</div></div>
                <div class="info-item"><div class="info-label">MCC</div><div class="info-value">{mcc:.6f}</div></div>
                <div class="info-item"><div class="info-label">Log Loss Norm</div><div class="info-value">{log_loss_norm:.6f}</div></div>
                <div class="info-item"><div class="info-label">Score Final</div><div class="info-value">{score_final:.6f}</div></div>
                <div class="info-item"><div class="info-label">Diferença Neg. Log-Veross.</div><div class="info-value">{diferenca_neg_log_veross:.6f}</div></div>
                <div class="info-item"><div class="info-label">Raio Elipsoide</div><div class="info-value">{raio_elipsoide:.2f}</div></div>
            </div>
        </div>

        {html_metricas_rank}

        {html_melhor}

        {html_medio}

        {html_ideal}

        <section class="plot-card">
            {fig_plotly_to_html_com_titulo(
                fig_classe_real,
                f"Dispersão 3D Interativa por Classe Real - {feature_1} x {feature_2} x {feature_3}",
                filename=f"{prefixo}_dispersao_3d"
            )}
        </section>

        <section class="plot-card">
            {fig_plotly_to_html_com_titulo(
                fig_boxplots,
                f"Boxplots das Features por Classe - {feature_1}, {feature_2} e {feature_3}",
                filename=f"{prefixo}_boxplots"
            )}
        </section>

        <section class="plot-card">
            {fig_plotly_to_html_com_titulo(
                fig_elipsoide_melhor,
                f"Elipsoide GMM de Não Fraude - Melhor Ponto de Corte - {feature_1} x {feature_2} x {feature_3}",
                filename=f"{prefixo}_elipsoide_melhor_corte"
            )}
        </section>

        <section class="plot-card">
            {fig_plotly_to_html_com_titulo(
                fig_elipsoide_medio,
                f"Elipsoide GMM de Não Fraude - Ponto de Corte Médio - {feature_1} x {feature_2} x {feature_3}",
                filename=f"{prefixo}_elipsoide_corte_medio"
            )}
        </section>

        <section class="plot-card">
            {fig_plotly_to_html_com_titulo(
                fig_spearman,
                f"Correlação de Spearman entre Features e Target - {feature_1}, {feature_2}, {feature_3} e Fraude",
                filename=f"{prefixo}_spearman"
            )}
        </section>

        <section class="plot-card">
            {fig_plotly_to_html_com_titulo(
                fig_projecoes_melhor,
                "Responsabilidades Estimadas pela GMM - Melhor Ponto de Corte - Projeções 2D",
                filename=f"{prefixo}_responsabilidades_melhor_corte"
            )}
        </section>

        <section class="plot-card">
            {fig_plotly_to_html_com_titulo(
                fig_projecoes_medio,
                "Responsabilidades Estimadas pela GMM - Ponto de Corte Médio - Projeções 2D",
                filename=f"{prefixo}_responsabilidades_corte_medio"
            )}
        </section>
    </section>
    """

    return {
        "rank": rank,
        "feature_1": feature_1,
        "feature_2": feature_2,
        "feature_3": feature_3,
        "html": html_rank,
        "tabela_metricas": tabela_metricas_rank,
        "tabela_matrizes": tabela_matrizes_rank
    }


# ============================================================
# FUNÇÃO PRINCIPAL UNIFICADA
# ============================================================

def gerar_relatorio_3x3_unificado(
    ranks=RANKS,
    arquivo_dados=ARQUIVO_DADOS,
    arquivo_scores=ARQUIVO_SCORES,
    pasta_saida=PASTA_SAIDA,
    nome_arquivo_html=ARQUIVO_HTML_SAIDA,
    pasta_latex=PASTA_LATEX,
    exportar_latex=True,
    raio_elipsoide=RAIO_ELIPSOIDE
):
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminho_html = pasta_saida / nome_arquivo_html
    caminho_latex = pasta_saida / pasta_latex
    caminho_latex.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(arquivo_dados)
    scores_3x3 = pd.read_csv(arquivo_scores)

    if "status_fraude" in df.columns:
        target_name = "status_fraude"
    elif "Class" in df.columns:
        df = df.rename(columns={"Class": "status_fraude"})
        target_name = "status_fraude"
    else:
        raise ValueError("Não encontrei a coluna target: 'status_fraude' ou 'Class'.")

    resultados = []

    for rank in ranks:
        resultado_rank = processar_rank_3x3(
            rank=rank,
            df=df,
            scores_3x3=scores_3x3,
            target_name=target_name,
            pasta_latex=caminho_latex,
            raio_elipsoide=raio_elipsoide
        )
        resultados.append(resultado_rank)

    tabela_metricas_total = pd.concat(
        [r["tabela_metricas"] for r in resultados],
        ignore_index=True
    )

    tabela_matrizes_total = pd.concat(
        [r["tabela_matrizes"] for r in resultados],
        ignore_index=True
    )

    if exportar_latex:
        exportar_tabelas_latex(
            pasta_latex=caminho_latex,
            tabela_metricas=tabela_metricas_total,
            tabela_matrizes=tabela_matrizes_total
        )

    tabela_metricas_total_html = tabela_metricas_total.copy()
    tabela_matrizes_total_html = tabela_matrizes_total.copy()

    for df_fmt in [tabela_metricas_total_html, tabela_matrizes_total_html]:
        for col in df_fmt.columns:
            if pd.api.types.is_float_dtype(df_fmt[col]):
                df_fmt[col] = df_fmt[col].apply(lambda x: formatar_float_html(x, 6))

    html_resumo_metricas = gerar_secao_tabela(
        titulo="Tabela Geral de Métricas dos Ranks",
        tabela_html=gerar_tabela_html(tabela_metricas_total_html, "tabela_metricas_ranks"),
        table_id="tabela_metricas_ranks",
        nome_csv="tabela_metricas_ranks.csv"
    )

    html_resumo_matrizes = gerar_secao_tabela(
        titulo="Tabela Geral das Matrizes de Confusão dos Ranks",
        tabela_html=gerar_tabela_html(tabela_matrizes_total_html, "tabela_matrizes_confusao_ranks"),
        table_id="tabela_matrizes_confusao_ranks",
        nome_csv="tabela_matrizes_confusao_ranks.csv"
    )

    navegacao_links = "\n".join([
        f'<a href="#rank-{r["rank"]}">Rank {r["rank"]} - {html.escape(str(r["feature_1"]))} + {html.escape(str(r["feature_2"]))} + {html.escape(str(r["feature_3"]))}</a>'
        for r in resultados
    ])

    html_ranks = "\n".join([r["html"] for r in resultados])

    html_final = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <title>Relatório 3x3 - Top Ranks</title>

        <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

        <style>
            body {{
                font-family: Arial, Helvetica, sans-serif;
                background: #f4f6f8;
                color: #020617;
                margin: 0;
                padding: 32px;
            }}

            .container {{
                max-width: 1500px;
                margin: 0 auto;
            }}

            h1 {{
                text-align: center;
                margin-bottom: 28px;
                color: #020617;
            }}

            .main-title {{
                font-size: 34px;
                margin-bottom: 14px;
            }}

            .nav-box {{
                background: #ffffff;
                border-radius: 16px;
                padding: 18px;
                margin-bottom: 28px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
                display: flex;
                justify-content: center;
                gap: 12px;
                flex-wrap: wrap;
            }}

            .nav-box a {{
                text-decoration: none;
                background: #eff6ff;
                color: #1e3a8a;
                border: 1px solid #bfdbfe;
                padding: 8px 12px;
                border-radius: 999px;
                font-weight: 800;
                font-size: 13px;
            }}

            .rank-section {{
                margin-top: 46px;
                padding-top: 12px;
                border-top: 4px solid #cbd5e1;
            }}

            .info-box {{
                background: #ffffff;
                border-radius: 16px;
                padding: 20px 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .info-grid {{
                display: grid;
                grid-template-columns: repeat(3, 1fr);
                gap: 14px;
                margin-top: 14px;
            }}

            .info-item {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                padding: 12px 14px;
            }}

            .info-label {{
                font-size: 13px;
                font-weight: 700;
                color: #475569;
                margin-bottom: 6px;
            }}

            .info-value {{
                font-size: 18px;
                font-weight: 800;
                color: #020617;
                font-family: Consolas, Monaco, monospace;
            }}

            .matrix-card,
            .plot-card {{
                background: white;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
                overflow-x: auto;
            }}

            .matrix-card h2,
            .plot-card h2 {{
                text-align: center;
                margin-top: 0;
                margin-bottom: 24px;
                color: #020617;
                font-size: 22px;
            }}

            .section-header {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 14px;
                flex-wrap: wrap;
                margin-bottom: 18px;
            }}

            .section-header h2 {{
                margin: 0;
            }}

            .section-actions-only {{
                display: flex;
                justify-content: flex-end;
                margin-bottom: 12px;
            }}

            .download-btn {{
                border: 1px solid #bfdbfe;
                background: #eff6ff;
                color: #1e3a8a;
                padding: 8px 12px;
                border-radius: 10px;
                font-weight: 800;
                font-size: 13px;
                cursor: pointer;
                text-decoration: none;
                display: inline-block;
            }}

            .download-btn:hover {{
                background: #dbeafe;
            }}

            .table-wrapper {{
                overflow-x: auto;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                max-height: 580px;
                overflow-y: auto;
            }}

            .data-table {{
                width: 100%;
                border-collapse: collapse;
                font-size: 13px;
                margin-top: 0;
            }}

            .data-table th {{
                background: #0f172a;
                color: white;
                padding: 10px 8px;
                text-align: left;
                position: sticky;
                top: 0;
                z-index: 1;
            }}

            .data-table td {{
                border-bottom: 1px solid #e2e8f0;
                padding: 8px;
                color: #020617;
                white-space: nowrap;
            }}

            .data-table tr:nth-child(even) {{
                background: #f8fafc;
            }}

            .matrix-area {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 34px;
            }}

            .matrix-wrapper {{
                display: grid;
                grid-template-columns: 180px 1fr 1fr;
                grid-template-rows: 48px 190px 190px;
                width: 950px;
            }}

            .corner {{
                background: transparent;
            }}

            .x-label {{
                display: flex;
                align-items: center;
                justify-content: center;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
                border-bottom: 1px solid #e5e7eb;
            }}

            .y-label {{
                display: flex;
                align-items: center;
                justify-content: flex-end;
                padding-right: 18px;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
            }}

            .cell {{
                display: flex;
                flex-direction: column;
                align-items: center;
                justify-content: center;
                min-height: 180px;
                border: 1px solid #e5e7eb;
                font-size: 20px;
                text-align: center;
                color: #020617 !important;
            }}

            .pct {{
                font-size: 30px;
                font-weight: 900;
                margin-bottom: 4px;
                color: #020617 !important;
            }}

            .count {{
                font-size: 24px;
                font-weight: 900;
                margin-bottom: 8px;
                color: #020617 !important;
            }}

            .cell-desc {{
                font-size: 13px;
                font-weight: 700;
                opacity: 1;
                color: #020617 !important;
            }}

            .q95 {{ background: #08306b; }}
            .q85 {{ background: #08519c; }}
            .q70 {{ background: #2171b5; }}
            .q50 {{ background: #6baed6; }}
            .q30 {{ background: #c6dbef; }}
            .q10 {{ background: #eff6ff; }}

            .legend {{
                position: relative;
                display: flex;
                flex-direction: column;
                align-items: center;
                min-width: 115px;
            }}

            .legend-title {{
                font-weight: 800;
                font-size: 15px;
                margin-bottom: 10px;
                color: #020617;
            }}

            .colorbar {{
                width: 30px;
                height: 310px;
                border-radius: 16px;
                background: linear-gradient(
                    to bottom,
                    #08306b 0%,
                    #08519c 18%,
                    #2171b5 36%,
                    #6baed6 58%,
                    #c6dbef 78%,
                    #eff6ff 100%
                );
                border: 1px solid #cbd5e1;
            }}

            .legend-label-top {{
                position: absolute;
                top: 43px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .legend-label-bottom {{
                position: absolute;
                top: 335px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .plotly-graph-div {{
                width: 100% !important;
            }}

            @media (max-width: 1100px) {{
                .info-grid {{
                    grid-template-columns: repeat(2, 1fr);
                }}

                .matrix-area {{
                    flex-direction: column;
                }}

                .matrix-wrapper {{
                    width: 100%;
                    grid-template-columns: 150px 1fr 1fr;
                }}
            }}

            @media (max-width: 700px) {{
                body {{
                    padding: 16px;
                }}

                .info-grid {{
                    grid-template-columns: 1fr;
                }}

                .matrix-wrapper {{
                    grid-template-columns: 120px 1fr 1fr;
                    grid-template-rows: 48px 160px 160px;
                }}

                .pct {{
                    font-size: 22px;
                }}

                .count {{
                    font-size: 18px;
                }}

                .y-label,
                .x-label {{
                    font-size: 14px;
                }}
            }}
        </style>
    </head>

    <body>
        <div class="container">

            <h1 class="main-title">Relatório Unificado 3x3 - Top Ranks</h1>

            <div class="nav-box">
                {navegacao_links}
            </div>

            {html_ranks}

            {html_resumo_metricas}

            {html_resumo_matrizes}

        </div>

        <script>
            function limparTextoCSV(texto) {{
                if (texto === null || texto === undefined) {{
                    return "";
                }}

                texto = String(texto).replace(/\\n/g, " ").replace(/\\s+/g, " ").trim();

                if (texto.includes(";") || texto.includes('"')) {{
                    texto = '"' + texto.replace(/"/g, '""') + '"';
                }}

                return texto;
            }}

            function baixarTabelaCSV(tableId, filename) {{
                const tabela = document.getElementById(tableId);

                if (!tabela) {{
                    alert("Tabela não encontrada: " + tableId);
                    return;
                }}

                const linhas = [];

                tabela.querySelectorAll("tr").forEach(function(row) {{
                    const celulas = Array.from(row.querySelectorAll("th, td"));
                    const linha = celulas.map(celula => limparTextoCSV(celula.innerText)).join(";");
                    linhas.push(linha);
                }});

                const csv = "\\ufeff" + linhas.join("\\n");
                const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
                const url = URL.createObjectURL(blob);

                const link = document.createElement("a");
                link.href = url;
                link.download = filename;
                document.body.appendChild(link);
                link.click();
                document.body.removeChild(link);

                URL.revokeObjectURL(url);
            }}
        </script>
    </body>
    </html>
    """

    caminho_html.write_text(html_final, encoding="utf-8")

    print("=" * 80)
    print("RELATÓRIO 3x3 UNIFICADO GERADO COM SUCESSO")
    print("=" * 80)
    print(f"HTML salvo em: {caminho_html.resolve()}")

    if exportar_latex:
        print(f"Arquivos LaTeX/imagens salvos em: {caminho_latex.resolve()}")

    print("=" * 80)

    return {
        "caminho_html": caminho_html,
        "caminho_latex": caminho_latex,
        "tabela_metricas": tabela_metricas_total,
        "tabela_matrizes": tabela_matrizes_total
    }


# ============================================================
# CHAMADA
# ============================================================

resultado_3x3_visu_scores = gerar_relatorio_3x3_unificado(
    ranks=(1, 2, 3),
    arquivo_dados="creditcard.csv",
    arquivo_scores="3x3_scores.csv",
    pasta_saida=".",
    nome_arquivo_html="3x3_visu_scores.html",
    pasta_latex="3x3_visu_scores",
    exportar_latex=True,
    raio_elipsoide=3.0
)

resultado_3x3_visu_scores["caminho_html"]


[Aviso] Não foi possível exportar rank_1_V11__V17__V26_dispersao_3d como PNG/PDF via Plotly/kaleido: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

[Aviso] Não foi possível exportar rank_1_V11__V17__V26_boxplots como PNG/PDF via Plotly/kaleido: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

[Aviso] Não foi possível exportar rank_1_V11__V17__V26_spearman como PNG/PDF via Plotly/kaleido: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

[Aviso] Não foi possível exportar rank_1_V11__V17__V26_elipsoide_melhor_corte como PNG/PDF via Plotly/kaleido: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

[Aviso] Não foi possív

WindowsPath('3x3_visu_scores.html')